In [77]:
# pip install pydicom


In [46]:
# Multi_Label_Classification.ipynb

import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import vision_models.constants as constants
from pydicom.pixel_data_handlers.util import apply_modality_lut
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from keras import layers, models, applications, losses

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


In [47]:
# Set paths
train_images_path = constants.TRAIN_DATA_PATH
train_df = pd.read_csv("NEW_EDA/train_dataset.csv")
val_df = pd.read_csv("NEW_EDA/val_dataset.csv")
test_df = pd.read_csv("NEW_EDA/test_dataset.csv")

# One-hot encode the 'condition' and 'level' columns separately
train_condition_dummies = pd.get_dummies(train_df['condition'], prefix='condition')
train_level_dummies = pd.get_dummies(train_df['level'], prefix='level')

val_condition_dummies = pd.get_dummies(val_df['condition'], prefix='condition')
val_level_dummies = pd.get_dummies(val_df['level'], prefix='level')

test_condition_dummies = pd.get_dummies(test_df['condition'], prefix='condition')
test_level_dummies = pd.get_dummies(test_df['level'], prefix='level')

# Concatenate the dummy columns back to the original DataFrame
train_df = pd.concat([train_df.drop(['condition', 'level'], axis=1), train_condition_dummies, train_level_dummies], axis=1)
val_df = pd.concat([val_df.drop(['condition', 'level'], axis=1), val_condition_dummies, val_level_dummies], axis=1)
test_df = pd.concat([test_df.drop(['condition', 'level'], axis=1), test_condition_dummies, test_level_dummies], axis=1)

# Now you have your DataFrame with one-hot encoded columns for 'condition' and 'level'


In [48]:
train_df = train_df.sample(frac=0.0005, random_state=42)
val_df = val_df.sample(frac=0.0005, random_state=42)
test_df = test_df.sample(frac=0.0005, random_state=42)

train_df

,study_id,series_id,image_name,file_meta_version,sop_class_uid,sop_instance_uid,transfer_syntax_uid,implementation_class_uid,implementation_version_name,content_date,...,condition_Left Neural Foraminal Narrowing,condition_Left Subarticular Stenosis,condition_Right Neural Foraminal Narrowing,condition_Right Subarticular Stenosis,condition_Spinal Canal Stenosis,level_L1/L2,level_L2/L3,level_L3/L4,level_L4/L5,level_L5/S1
29239,1820866003,3689221841,21.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1820866003.1.21,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,True,False,False,False,False,False,False,False,True
10576,3084269121,1616142642,3.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3084269121.1.3,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False
38534,4173917544,111096887,13.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,4173917544.1.13,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False
40490,2905685162,1535449979,14.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,2905685162.1.14,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False
115633,861719444,3451308655,87.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,861719444.1.87,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49099,1302048123,272639520,38.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1302048123.1.38,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,True,False,False,False,False,False,False,True,False
131137,3745670967,615061872,44.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3745670967.1.44,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False
6818,1780646606,3098929855,21.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1780646606.1.21,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,True,False,True,False,False,False,False
39227,3647070644,1825894086,23.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3647070644.1.23,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,False


In [49]:


# Data Preparation
for df in [train_df, val_df, test_df]:
    df['image_path'] = df.apply(lambda row: os.path.join(train_images_path, str(row['study_id']), str(row['series_id']), f"{row['instance_number']}.dcm"), axis=1)


In [50]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(69, 47)
(8, 47)
(9, 47)


In [56]:
# Keep only the necessary columns for the multi-label classification task
columns_to_keep = [
    'image_path',
    'condition_Left Neural Foraminal Narrowing',
    'condition_Left Subarticular Stenosis',
    'condition_Right Neural Foraminal Narrowing',
    'condition_Right Subarticular Stenosis',
    'condition_Spinal Canal Stenosis',
    'level_L1/L2',
    'level_L2/L3',
    'level_L3/L4',
    'level_L4/L5',
    'level_L5/S1'
]

# Drop all other columns
train_df = train_df[columns_to_keep]
val_df = val_df[columns_to_keep]
test_df = test_df[columns_to_keep]

print(train_df.columns)  # Check the remaining columns to ensure correctness
print(val_df.columns)
print(test_df.columns)


Index(['image_path', 'condition_Left Neural Foraminal Narrowing',
       'condition_Left Subarticular Stenosis',
       'condition_Right Neural Foraminal Narrowing',
       'condition_Right Subarticular Stenosis',
       'condition_Spinal Canal Stenosis', 'level_L1/L2', 'level_L2/L3',
       'level_L3/L4', 'level_L4/L5', 'level_L5/S1'],
      dtype='object')
Index(['image_path', 'condition_Left Neural Foraminal Narrowing',
       'condition_Left Subarticular Stenosis',
       'condition_Right Neural Foraminal Narrowing',
       'condition_Right Subarticular Stenosis',
       'condition_Spinal Canal Stenosis', 'level_L1/L2', 'level_L2/L3',
       'level_L3/L4', 'level_L4/L5', 'level_L5/S1'],
      dtype='object')
Index(['image_path', 'condition_Left Neural Foraminal Narrowing',
       'condition_Left Subarticular Stenosis',
       'condition_Right Neural Foraminal Narrowing',
       'condition_Right Subarticular Stenosis',
       'condition_Spinal Canal Stenosis', 'level_L1/L2', 'level_

In [57]:

# Function to read and preprocess images using pydicom
def load_image(img_path, target_size=(224, 224)):
    # Load DICOM file
    dicom = pydicom.dcmread(img_path.numpy().decode('utf-8'))
    img = dicom.pixel_array
    
    # Normalize the image
    img = img / np.max(img)
    
    # Ensure img has 3 dimensions (Height, Width, Channels)
    if len(img.shape) == 2:  # Grayscale image
        img = np.expand_dims(img, axis=-1)  # Add channel dimension
    
    # Convert grayscale to RGB (3 channels) if needed
    if img.shape[-1] == 1:
        img = np.concatenate([img, img, img], axis=-1)
    
    # Convert to float32 immediately
    img = tf.convert_to_tensor(img, dtype=tf.float32)
    
    # Now resize the image
    img = tf.image.resize(img, target_size)
    
    # Normalize pixel values to [0, 1]
    img = img / 255.0
    
    return img

# Wrapper function to use with tf.data.Dataset.map
def load_image_wrapper(img_path, target_size=(224, 224)):
    img = tf.py_function(func=load_image, inp=[img_path], Tout=tf.float32)
    img.set_shape((target_size[0], target_size[1], 3))  # Explicitly set shape (224, 224, 3)
    return img

# Create a TensorFlow dataset
def create_tf_dataset(df, batch_size=32, is_training=True, predict_condition=True, predict_level=True):
    # Dynamically choose label columns based on what you want to predict
    if predict_condition and predict_level:
        label_columns = [col for col in df.columns if col.startswith('condition_') or col.startswith('level_')]
    elif predict_condition:
        label_columns = [col for col in df.columns if col.startswith('condition_')]
    elif predict_level:
        label_columns = [col for col in df.columns if col.startswith('level_')]
    else:
        raise ValueError("At least one of `predict_condition` or `predict_level` must be True.")
    
    dataset = tf.data.Dataset.from_tensor_slices((df['image_path'], df[label_columns]))
    dataset = dataset.map(lambda x, y: (load_image_wrapper(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    
    # Debugging: Print shape of each element in the dataset
    def print_shapes(x, y):
        print(f"Image shape: {x.shape}, Label shape: {y.shape}")
        return x, y
    
    dataset = dataset.map(print_shapes)  # Add this line to print shapes
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# Example of creating datasets
train_dataset_condition = create_tf_dataset(train_df, predict_condition=True, predict_level=False)
train_dataset_level = create_tf_dataset(train_df, predict_condition=False, predict_level=True)
train_dataset_both = create_tf_dataset(train_df, predict_condition=True, predict_level=True)



Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (10,)


In [58]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import confusion_matrix, classification_report

# Ensure the directory exists
def ensure_directory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory created: {directory}")
    else:
        print(f"Directory already exists: {directory}")

# Plot training history
def plot_metrics(history, title):
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    save_path = f'NEW_EDA/{title}.png'
    ensure_directory(os.path.dirname(save_path))  # Ensure the directory exists
    plt.savefig(save_path)
    plt.close()  # Close the plot to ensure it's saved
    print(f"Training history saved at {save_path}")

# Plot confusion matrix and save it
def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix', cmap=plt.cm.Blues, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=classes, yticklabels=classes)
    
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    if save_path:
        ensure_directory(os.path.dirname(save_path))  # Ensure the directory exists
        plt.savefig(save_path)
        plt.close()  # Close the plot to ensure it's saved
        print(f"Confusion matrix saved at {save_path}")
    else:
        plt.show()

# Function to save predictions and actual labels to CSV
def save_predictions_to_csv(true_labels, predicted_labels, df, dataset_name, label_classes, model_type):
    # Map the indices back to their actual label names
    true_names = [label_classes[i] for i in true_labels]
    predicted_names = [label_classes[i] for i in predicted_labels]
    
    # Create a DataFrame with true and predicted labels
    results_df = pd.DataFrame({
        'image_path': df['image_path'],
        'true_label': true_names,
        'predicted_label': predicted_names
    })
    
    # Save to CSV
    save_path = f'NEW_EDA/{dataset_name}_{model_type}_predictions.csv'
    ensure_directory(os.path.dirname(save_path))  # Ensure the directory exists
    results_df.to_csv(save_path, index=False)
    print(f"{dataset_name}_{model_type}_predictions.csv saved successfully!")

# Evaluate the model and plot the confusion matrix
def evaluate_model(model, dataset, df, dataset_name, condition_classes=None, level_classes=None, task_type='condition', model_type=''):
    # Determine which label columns to use
    if task_type == 'condition':
        label_columns = [col for col in df.columns if col.startswith('condition_')]
        print("Condition columns used:", label_columns)
        predictions = model.predict(dataset)
        predicted_labels = np.argmax(predictions, axis=1)
        true_labels = np.argmax(df[label_columns].values, axis=1)
        evaluate_and_save(predicted_labels, true_labels, condition_classes, df, dataset_name, task_type, model_type)
    
    elif task_type == 'level':
        label_columns = [col for col in df.columns if col.startswith('level_')]
        print("Level columns used:", label_columns)
        predictions = model.predict(dataset)
        predicted_labels = np.argmax(predictions, axis=1)
        true_labels = np.argmax(df[label_columns].values, axis=1)
        evaluate_and_save(predicted_labels, true_labels, level_classes, df, dataset_name, task_type, model_type)
    
    elif task_type == 'condition_level':
        condition_columns = [col for col in df.columns if col.startswith('condition_')]
        level_columns = [col for col in df.columns if col.startswith('level_')]
        print("Condition columns used:", condition_columns)
        print("Level columns used:", level_columns)
        
        predictions = model.predict(dataset)
        condition_predicted_labels = np.argmax(predictions[0], axis=1)
        level_predicted_labels = np.argmax(predictions[1], axis=1)
        
        condition_true_labels = np.argmax(df[condition_columns].values, axis=1)
        level_true_labels = np.argmax(df[level_columns].values, axis=1)
        
        evaluate_and_save(condition_predicted_labels, condition_true_labels, condition_classes, df, dataset_name, 'condition', model_type)
        evaluate_and_save(level_predicted_labels, level_true_labels, level_classes, df, dataset_name, 'level', model_type)

def evaluate_and_save(predicted_labels, true_labels, label_classes, df, dataset_name, task_type='condition', model_type=''):
    # Check the unique classes in both true and predicted labels
    unique_true_labels = set(true_labels)
    unique_pred_labels = set(predicted_labels)
    
    print(f"Unique true {task_type} labels: {unique_true_labels}")
    print(f"Unique predicted {task_type} labels: {unique_pred_labels}")
    
    print(f"Evaluation on {dataset_name} Data ({task_type.title()}):")
    try:
        print(classification_report(true_labels, predicted_labels, target_names=label_classes))
    except ValueError as e:
        print(f"Error during classification report generation: {e}")
        print(f"True labels: {true_labels}")
        print(f"Predicted labels: {predicted_labels}")
        print(f"Label classes: {label_classes}")
    
    # Plot confusion matrix
    save_path = f'NEW_EDA/{dataset_name}_{task_type}_{model_type}_confusion_matrix.png'
    plot_confusion_matrix(true_labels, predicted_labels, classes=label_classes, title=f'{dataset_name} {task_type.title()} Confusion Matrix ({model_type})', save_path=save_path)
    
    # Save predictions and actual labels to CSV
    save_predictions_to_csv(true_labels, predicted_labels, df, dataset_name + f'_{task_type}', label_classes, model_type)


# Function to create the CNN model
def create_cnn_model(task_type, num_condition_classes=None, num_level_classes=None):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu')
    ])
    
    if task_type == 'condition':
        model.add(layers.Dense(num_condition_classes, activation='softmax', name='condition_output'))
    elif task_type == 'level':
        model.add(layers.Dense(num_level_classes, activation='softmax', name='level_output'))
    elif task_type == 'condition_level':
        model.add(layers.Dense(num_condition_classes, activation='softmax', name='condition_output'))
        model.add(layers.Dense(num_level_classes, activation='softmax', name='level_output'))
    
    return model

# Function to create the ResNet model
def create_resnet_model(task_type, num_condition_classes=None, num_level_classes=None):
    base_model = applications.ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    
    condition_output = None
    level_output = None
    
    if task_type == 'condition':
        condition_output = layers.Dense(num_condition_classes, activation='softmax', name='condition_output')(x)
        model = models.Model(inputs=base_model.input, outputs=condition_output)
    elif task_type == 'level':
        level_output = layers.Dense(num_level_classes, activation='softmax', name='level_output')(x)
        model = models.Model(inputs=base_model.input, outputs=level_output)
    elif task_type == 'condition_level':
        condition_output = layers.Dense(num_condition_classes, activation='softmax', name='condition_output')(x)
        level_output = layers.Dense(num_level_classes, activation='softmax', name='level_output')(x)
        model = models.Model(inputs=base_model.input, outputs=[condition_output, level_output])
    
    return model

# Function to dynamically create and train a model
def run_model(task_type, model_type, train_df, val_df, test_df):
    num_condition_classes = len([col for col in train_df.columns if col.startswith('condition_')])
    num_level_classes = len([col for col in train_df.columns if col.startswith('level_')])

    if model_type == 'cnn':
        model = create_cnn_model(task_type, num_condition_classes, num_level_classes)
    elif model_type == 'resnet':
        model = create_resnet_model(task_type, num_condition_classes, num_level_classes)

    if task_type == 'condition':
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    elif task_type == 'level':
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    elif task_type == 'condition_level':
        model.compile(optimizer='adam', 
                      loss={'condition_output': 'categorical_crossentropy', 'level_output': 'categorical_crossentropy'},
                      metrics={'condition_output': 'accuracy', 'level_output': 'accuracy'})

    train_dataset = create_tf_dataset(train_df, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))
    val_dataset = create_tf_dataset(val_df, is_training=False, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))
    test_dataset = create_tf_dataset(test_df, is_training=False, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))

    history = model.fit(train_dataset, validation_data=val_dataset, epochs=2)
    
    title_suffix = f"{model_type.upper()} Model - {task_type.replace('_', ' & ').title()}"
    plot_metrics(history, f"Training History for {title_suffix}")

    # Evaluate and save predictions for validation and test datasets
    condition_classes = [col for col in val_df.columns if col.startswith('condition_')]
    level_classes = [col for col in val_df.columns if col.startswith('level_')]

    if task_type == 'condition_level':
        evaluate_model(model, val_dataset, val_df, "Validation", condition_classes, level_classes, task_type, model_type)
        evaluate_model(model, test_dataset, test_df, "Test", condition_classes, level_classes, task_type, model_type)
    else:
        label_classes = condition_classes if task_type == 'condition' else level_classes
        evaluate_model(model, val_dataset, val_df, "Validation", label_classes, task_type, model_type)
        evaluate_model(model, test_dataset, test_df, "Test", label_classes, task_type, model_type)

# Example usage:
# run_model('level', 'cnn', train_df, val_df, test_df)  # Predict condition using CNN


In [60]:


# Example usage:
# run_model('condition', 'cnn', train_df, val_df, test_df)  # Predict condition using CNN
run_model('condition', 'resnet', train_df, val_df, test_df)    # Predict level using ResNet

run_model('level', 'cnn', train_df, val_df, test_df)  # Predict condition using CNN
run_model('level', 'resnet', train_df, val_df, test_df)    # Predict level using ResNet

# run_model('condition_level', 'cnn', train_df, val_df, test_df)  # Predict both condition and level using CNN
# run_model('condition_level', 'resnet', train_df, val_df, test_df)  # Predict both condition and level using CNN


Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.5416 - loss: 0.5148 - val_accuracy: 0.0000e+00 - val_loss: 0.1645
Epoch 2/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.0792 - loss: 0.4779 - val_accuracy: 0.0000e+00 - val_loss: 0.2257
Directory already exists: NEW_EDA
Training history saved at NEW_EDA/Training History for RESNET Model - Condition.png
Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 438ms/step - accuracy: 0.0262 - loss: 0.5101 - val_accuracy: 0.0000e+00 - val_loss: 0.2045
Epoch 2/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.0864 - loss: 0.4564 - val_accuracy: 0.1250 - val_loss: 0.1133
Directory already exists: NEW_EDA
Training history saved at NEW_EDA/Training History for CNN Mode